# Experiment 2 — native expression vs Python UDF

The same filter three ways. Run each timing cell **twice** and use the second time (the first includes warm-up).
Spark UI: http://localhost:4040 → SQL / DataFrame tab for the executed plans.

In [1]:
import time
import pandas as pd
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("m2-exp2-udf").getOrCreate()
orders = spark.read.parquet("/opt/data/m1/orders")

def timed(label, df):
    t0 = time.perf_counter()
    n = df.count()
    print(f"{label}: {n:,} rows in {time.perf_counter() - t0:.2f}s")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 13:24:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

## The three versions of `bucket(amount) == 3`, where bucket = floor(amount / 100)

**PREDICT** before running anything below:
1. Which extra operator(s) will the UDF plan contain that the native plan does not?
2. Will the `Scan parquet` node show the same *PushedFilters* in both?
3. Runtime: native vs Python UDF vs pandas UDF — order them, and guess the ratio.

In [3]:
native = orders.filter(F.floor(F.col("amount") / 100) == 3)

@F.udf("bigint")
def bucket_py(a):
    return int(a // 100)

python_udf = orders.filter(bucket_py("amount") == 3)

@F.pandas_udf("bigint")
def bucket_pd(a: pd.Series) -> pd.Series:
    return (a // 100).astype("int64")

pandas_udf = orders.filter(bucket_pd("amount") == 3)

## Plans

In [4]:
native.explain("formatted")

== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m1/orders]
PushedFilters: [IsNotNull(amount)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4]

(3) Filter [codegen id : 1]
Input [5]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4]
Condition : (isnotnull(amount#3) AND (FLOOR((amount#3 / 100.0)) = 3))




In [5]:
python_udf.explain("formatted")

== Physical Plan ==
* Project (5)
+- * Filter (4)
   +- BatchEvalPython (3)
      +- * ColumnarToRow (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m1/orders]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4]

(3) BatchEvalPython
Input [5]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4]
Arguments: [bucket_py(amount#3)#8L], [pythonUDF0#10L]

(4) Filter [codegen id : 2]
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, pythonUDF0#10L]
Condition : (pythonUDF0#10L = 3)

(5) Project [codegen id : 2]
Output [5]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4]
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, pythonUDF0#

In [6]:
pandas_udf.explain("formatted")

== Physical Plan ==
* Project (5)
+- * Filter (4)
   +- ArrowEvalPython (3)
      +- * ColumnarToRow (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m1/orders]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4]

(3) ArrowEvalPython
Input [5]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4]
Arguments: [bucket_pd(amount#3)#9L], [pythonUDF0#11L], 200

(4) Filter [codegen id : 2]
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, pythonUDF0#11L]
Condition : (pythonUDF0#11L = 3)

(5) Project [codegen id : 2]
Output [5]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4]
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, python

## Timing (run this cell twice)

In [8]:
timed("native    ", native)
timed("python udf", python_udf)
timed("pandas udf", pandas_udf)

native    : 400,848 rows in 0.17s
python udf: 400,848 rows in 0.47s


[Stage 16:>                                                         (0 + 8) / 8]

pandas udf: 400,848 rows in 0.76s


In [9]:
big = (spark.range(0, 20_000_000, numPartitions=8)
         .withColumn("amount", F.round(F.rand(seed=1) * 500 + 1, 2)))

timed("native    ", big.filter(F.floor(F.col("amount") / 100) == 3))
timed("python udf", big.filter(bucket_py("amount") == 3))
timed("pandas udf", big.filter(bucket_pd("amount") == 3))

native    : 4,001,315 rows in 0.58s


python udf: 4,001,315 rows in 2.05s


[Stage 25:=======>                                                  (1 + 7) / 8]

pandas udf: 4,001,315 rows in 0.94s


## Stop

In [ ]:
spark.stop()